# Demo: DriftInjector — drift sintético en PhysioNet

Pipeline completo:
1. Carga PhysioNet (train + ood_test como proxy de target)
2. Fit del `DriftInjector` en train
3. Aplica drift numérico (Beta mixture) sobre las 5 variables de signos vitales
4. Visualiza antes/después
5. Muestra tabla resumen con métricas KS y PSI

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from darl.data.get_dataset import load_dataset
from darl.drift import DriftInjector
from darl.visualization.drift_plots import plot_numeric_drift_grid


In [ ]:
print('Cargando PhysioNet...')
dset = load_dataset('physionet')

df_train, y_train, _, _ = dset.get_pandas('train')
df_target, y_target, _, _ = dset.get_pandas('ood_test')

print(f'Train : {df_train.shape}')
print(f'Target: {df_target.shape}')


In [ ]:
VITALS = ['HR', 'SBP', 'MAP', 'Resp', 'Temp']

inj = DriftInjector(random_state=42)
inj.fit(df_train, numeric_cols=VITALS)
print('DriftInjector fitted.')


In [ ]:
ALPHA = 0.3

# Dirección del drift por variable
num_cfg = {
    'HR':   'high',     # taquicardia
    'SBP':  'low',      # hipotensión
    'MAP':  'low',      # hipotensión media
    'Resp': 'high',     # taquipnea
    'Temp': 'extreme',  # temperaturas extremas
}

df_drifted, metadata = inj.transform(
    df_target,
    alpha=ALPHA,
    numeric_drift_config=num_cfg,
)
print(f'Drift aplicado con α={ALPHA}.')


In [ ]:
summary = DriftInjector.summary(metadata)
print(summary.to_string())
summary


In [ ]:
fig = plot_numeric_drift_grid(
    df_target, df_drifted, cols=VITALS, ncols=3, bins=60
)
plt.show()


## Efecto de distintas severidades α

In [ ]:
alphas = [0.1, 0.3, 0.6, 1.0]
results = []

for a in alphas:
    _, meta_a = inj.transform(df_target, alpha=a, numeric_drift_config=num_cfg)
    row = DriftInjector.summary(meta_a)[['severity_alpha', 'ks_stat', 'psi']].copy()
    row.insert(0, 'alpha', a)
    results.append(row)

pd.concat(results)
